In [35]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [36]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)

/Users/ashutosh/PycharmProjects/Housing_Prices_Regression_ML_End_to_End/venv/bin/python
3.2.0
/Users/ashutosh/PycharmProjects/Housing_Prices_Regression_ML_End_to_End/venv/lib/python3.12/site-packages/xgboost/__init__.py


In [37]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

#### Load the datasets which are feature-engineered and leakage-safe

In [38]:
train_df = pd.read_csv("/Users/ashutosh/PycharmProjects/Housing_Prices_Regression_ML_End_to_End/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("/Users/ashutosh/PycharmProjects/Housing_Prices_Regression_ML_End_to_End/data/processed/feature_engineered_eval.csv")

##### Declare the target feature and the other features

In [39]:
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


#### Define the loss function for Optuna with MLFlow

In [40]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

##### Run the Optuna study with the MLFlow

###### Constraining the MLFlow to always use the root project mlruns folder

In [41]:
mlflow.set_tracking_uri("/Users/ashutosh/PycharmProjects/Housing_Prices_Regression_ML_End_to_End/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

[I 2026-03-20 00:49:36,870] A new study created in memory with name: no-name-bdd20949-9d1d-41ec-b2d6-676be5067f79
[I 2026-03-20 00:49:47,073] Trial 0 finished with value: 68264.81056040667 and parameters: {'n_estimators': 464, 'max_depth': 8, 'learning_rate': 0.03669269625494924, 'subsample': 0.9443118352221329, 'colsample_bytree': 0.6449066226172375, 'min_child_weight': 2, 'gamma': 3.3816907065516544, 'reg_alpha': 5.797091389979948e-07, 'reg_lambda': 0.41055457981124394}. Best is trial 0 with value: 68264.81056040667.
[I 2026-03-20 00:50:01,894] Trial 1 finished with value: 68343.54532277148 and parameters: {'n_estimators': 731, 'max_depth': 8, 'learning_rate': 0.041874716857481675, 'subsample': 0.7027363021781908, 'colsample_bytree': 0.7246927665811971, 'min_child_weight': 3, 'gamma': 2.0054546844366388, 'reg_alpha': 7.056775938153766e-07, 'reg_lambda': 1.2972989568226567e-06}. Best is trial 0 with value: 68264.81056040667.
[I 2026-03-20 00:50:04,591] Trial 2 finished with value: 747

Best params: {'n_estimators': 701, 'max_depth': 9, 'learning_rate': 0.04017434867835078, 'subsample': 0.6363164855242762, 'colsample_bytree': 0.6746451245409215, 'min_child_weight': 9, 'gamma': 2.40976610955201, 'reg_alpha': 0.009789279004668494, 'reg_lambda': 9.658173336902487e-06}


#### Train the final model using the best hyperparameter combinations from Optuna

In [42]:
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MeanAbsoluteError:", mae)
print("RootMeanSquaredError:", rmse)
print("R²:", r2)

Final tuned model performance:
MeanAbsoluteError: 30618.570881665775
RootMeanSquaredError: 70977.2680414172
R²: 0.961068728386655


#### Log the final model in MLFlow

In [43]:
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"RootMeanSquaredError": rmse, "MeanAbsoluteError": mae, "R²": r2})
    mlflow.xgboost.log_model(best_model, name="model")